In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import norm
import dynesty
from dynesty import plotting as dyplot
import pandas as pd
import corner

# ============================================================================
# CONSTANTES FÍSICAS
# ============================================================================
c = 2.998e10  # cm/s
H0 = 67.4  # km/s/Mpc (Planck 2018)
Omega_m = 0.315
Omega_Lambda = 0.685
H0_cgs = H0 * 1e5 / 3.086e24  # s^-1

# ============================================================================
# COSMOLOGÍA
# ============================================================================
def E_z(z):
    """Normalized Hubble parameter"""
    return np.sqrt(Omega_m * (1 + z)**3 + Omega_Lambda)

def luminosity_distance(z):
    """Luminosity distance in cm"""
    def integrand(zp):
        return 1.0 / E_z(zp)

    integral, _ = quad(integrand, 0, z)
    D_L = (c / H0_cgs) * (1 + z) * integral
    return D_L

# ============================================================================
# MODELO LIV: TIME DELAY
# ============================================================================
def Delta_t_LIV_linear(E, z, E_QG):
    """
    Linear LIV time delay (n=1)
    Δt_LIV = ± (1+z) / H0 * ∫dz' [1/E(z')] * E / E_QG

    Parameters:
    -----------
    E : float or array
        Photon energy in GeV
    z : float
        Redshift
    E_QG : float
        Quantum gravity energy scale in GeV

    Returns:
    --------
    float or array
        Time delay in seconds
    """
    def integrand(zp):
        return 1.0 / E_z(zp)

    integral, _ = quad(integrand, 0, z)

    # Convert E from GeV to erg: 1 GeV = 1.602e-3 erg
    delta_t = (1 + z) * integral / H0_cgs * (E * 1e9) / (E_QG * 1e9)  # eV units

    return delta_t

def Delta_t_LIV_quadratic(E, z, E_QG):
    """
    Quadratic LIV time delay (n=2)
    Δt_LIV ∝ E^2 / E_QG^2
    """
    def integrand(zp):
        return 1.0 / E_z(zp)

    integral, _ = quad(integrand, 0, z)
    delta_t = (1 + z) * integral / H0_cgs * (E / E_QG)**2

    return delta_t

# ============================================================================
# MODELO INTRÍNSECO: SMOOTH BROKEN POWER-LAW (SBPL)
# ============================================================================
def Delta_t_intrinsic_SBPL(E, zeta, E_b, alpha1, alpha2, mu):
    """
    Smoothly broken power-law intrinsic delay

    Parameters:
    -----------
    E : float or array
        Energy in GeV
    zeta : float
        Normalization constant (seconds)
    E_b : float
        Break energy (GeV)
    alpha1 : float
        Power-law index before break
    alpha2 : float
        Power-law index after break
    mu : float
        Smoothness parameter (transition width)

    Returns:
    --------
    float or array
        Intrinsic time delay in seconds
    """
    x = E / E_b
    term1 = x**alpha1
    term2 = x**alpha2

    # Smooth transition
    f = 1.0 / (1.0 + x**mu)

    delta_t_int = zeta * (f * term1 + (1 - f) * term2)

    return delta_t_int

def Delta_t_intrinsic_simple_PL(E, tau_0, beta):
    """
    Simple power-law intrinsic delay
    Δt_int = τ_0 * (E / 1 GeV)^β
    """
    return tau_0 * E**beta

# ============================================================================
# MODELO TOTAL
# ============================================================================
def time_delay_model(E, z, params, model_type='LIV_linear_SBPL'):
    """
    Total observed time delay

    Parameters:
    -----------
    E : array
        Photon energies in GeV
    z : float
        Redshift
    params : dict
        Model parameters
    model_type : str
        'LIV_linear_SBPL', 'LIV_linear_PL', 'LIV_quadratic_SBPL', etc.
    """
    if model_type == 'LIV_linear_SBPL':
        E_QG = params['E_QG']
        zeta = params['zeta']
        E_b = params['E_b']
        alpha1 = params['alpha1']
        alpha2 = params['alpha2']
        mu = params['mu']
        sign = params['sign']  # ±1

        Delta_LIV = sign * Delta_t_LIV_linear(E, z, E_QG)
        Delta_int = Delta_t_intrinsic_SBPL(E, zeta, E_b, alpha1, alpha2, mu)

        return Delta_LIV + Delta_int

    elif model_type == 'intrinsic_only_SBPL':
        zeta = params['zeta']
        E_b = params['E_b']
        alpha1 = params['alpha1']
        alpha2 = params['alpha2']
        mu = params['mu']

        return Delta_t_intrinsic_SBPL(E, zeta, E_b, alpha1, alpha2, mu)

    elif model_type == 'LIV_linear_PL':
        E_QG = params['E_QG']
        tau_0 = params['tau_0']
        beta = params['beta']
        sign = params['sign']

        Delta_LIV = sign * Delta_t_LIV_linear(E, z, E_QG)
        Delta_int = Delta_t_intrinsic_simple_PL(E, tau_0, beta)

        return Delta_LIV + Delta_int

# ============================================================================
# LIKELIHOOD FUNCTION
# ============================================================================
def log_likelihood(theta, E_obs, t_obs, sigma_t, z, model_type):
    """
    Log-likelihood for observed time delays

    Parameters:
    -----------
    theta : array
        Parameter vector
    E_obs : array
        Observed photon energies (GeV)
    t_obs : array
        Observed arrival times (seconds relative to trigger)
    sigma_t : array
        Time uncertainties (seconds)
    z : float
        Redshift
    model_type : str
        Model specification
    """
    # Parse parameters based on model
    if model_type == 'LIV_linear_SBPL':
        E_QG, zeta, E_b, alpha1, alpha2, mu, sign_raw = theta
        sign = 1 if sign_raw > 0.5 else -1
        params = {'E_QG': 10**E_QG, 'zeta': zeta, 'E_b': E_b,
                  'alpha1': alpha1, 'alpha2': alpha2, 'mu': mu, 'sign': sign}

    elif model_type == 'intrinsic_only_SBPL':
        zeta, E_b, alpha1, alpha2, mu = theta
        params = {'zeta': zeta, 'E_b': E_b, 'alpha1': alpha1,
                  'alpha2': alpha2, 'mu': mu}

    # Compute model predictions
    t_model = time_delay_model(E_obs, z, params, model_type)

    # Compute residuals
    residuals = t_obs - t_model

    # Gaussian likelihood
    log_L = -0.5 * np.sum((residuals / sigma_t)**2 + np.log(2 * np.pi * sigma_t**2))

    return log_L

# ============================================================================
# PRIOR TRANSFORM (for dynesty)
# ============================================================================
def prior_transform_LIV_SBPL(u):
    """
    Transform unit cube to parameter space for LIV + SBPL model

    Priors (conservative based on literature):
    - log10(E_QG/GeV): Uniform[15, 20]
    - zeta: Uniform[-10, 10] seconds
    - E_b: Log-Uniform[0.01, 100] GeV
    - alpha1: Uniform[-3, 3]
    - alpha2: Uniform[-3, 3]
    - mu: Uniform[0.1, 10]
    - sign: Uniform[0, 1] (mapped to ±1 in likelihood)
    """
    params = np.empty(7)

    # log10(E_QG): Uniform[15, 20]
    params[0] = 15.0 + 5.0 * u[0]

    # zeta: Uniform[-10, 10]
    params[1] = -10.0 + 20.0 * u[1]

    # E_b: Log-Uniform[0.01, 100]
    params[2] = 10**(np.log10(0.01) + (np.log10(100) - np.log10(0.01)) * u[2])

    # alpha1: Uniform[-3, 3]
    params[3] = -3.0 + 6.0 * u[3]

    # alpha2: Uniform[-3, 3]
    params[4] = -3.0 + 6.0 * u[4]

    # mu: Uniform[0.1, 10]
    params[5] = 0.1 + 9.9 * u[5]

    # sign: Uniform[0, 1]
    params[6] = u[6]

    return params

def prior_transform_intrinsic_SBPL(u):
    """Prior transform for intrinsic-only SBPL"""
    params = np.empty(5)

    params[0] = -10.0 + 20.0 * u[0]  # zeta
    params[1] = 10**(np.log10(0.01) + (np.log10(100) - np.log10(0.01)) * u[1])  # E_b
    params[2] = -3.0 + 6.0 * u[2]  # alpha1
    params[3] = -3.0 + 6.0 * u[3]  # alpha2
    params[4] = 0.1 + 9.9 * u[4]  # mu

    return params

# ============================================================================
# DATASET REAL: GRB 090510 (EJEMPLO CON DATOS PUBLICADOS)
# ============================================================================
"""
GRB 090510 (Short GRB):
- Redshift: z = 0.903 (Roming et al. 2009)
- Highest energy photon: 31 GeV
- Time delay from trigger: 0.86 seconds
- Publicado en literatura como constraint tight para LIV
"""

# Photon data simulado basado en distribución típica LAT
# (En análisis real, esto vendría de archivos FITS)
def generate_GRB090510_mock_data():
    """
    Generate mock photon data for GRB 090510 based on published info
    """
    np.random.seed(42)

    # Energies: log-uniform distribution from 0.1 to 31 GeV
    N_photons = 50
    E_obs = 10**(np.random.uniform(np.log10(0.1), np.log10(31), N_photons))

    # Sort by energy
    E_obs = np.sort(E_obs)

    # Arrival times: intrinsic delay + noise
    # Assuming simple power-law intrinsic: t ∝ E^(-0.5) + noise
    t_intrinsic = 2.0 * (E_obs / 1.0)**(-0.5)  # seconds
    t_obs = t_intrinsic + np.random.normal(0, 0.1, N_photons)

    # Time uncertainties (typical LAT timing precision ~100 μs, dominated by other systematics)
    sigma_t = np.full(N_photons, 0.05)  # 50 ms uncertainty

    z = 0.903

    return E_obs, t_obs, sigma_t, z

# ============================================================================
# NESTED SAMPLING ANALYSIS
# ============================================================================
def run_nested_sampling(E_obs, t_obs, sigma_t, z, model_type='LIV_linear_SBPL',
                        nlive=500, dlogz=0.01):
    """
    Run nested sampling to compute Bayesian evidence and posteriors
    """
    print(f"\n{'='*70}")
    print(f"Running Nested Sampling: {model_type}")
    print(f"{'='*70}")
    print(f"N_photons: {len(E_obs)}")
    print(f"Redshift: {z:.3f}")
    print(f"Energy range: {E_obs.min():.2f} - {E_obs.max():.2f} GeV")

    # Select appropriate prior transform
    if model_type == 'LIV_linear_SBPL':
        prior_transform = prior_transform_LIV_SBPL
        ndim = 7
    elif model_type == 'intrinsic_only_SBPL':
        prior_transform = prior_transform_intrinsic_SBPL
        ndim = 5

    # Define loglikelihood wrapper
    def loglike(theta):
        return log_likelihood(theta, E_obs, t_obs, sigma_t, z, model_type)

    # Run dynesty
    sampler = dynesty.NestedSampler(loglike, prior_transform, ndim,
                                     nlive=nlive, bound='multi', sample='rwalk')

    sampler.run_nested(dlogz=dlogz, print_progress=True)
    results = sampler.results

    # Extract evidence
    logZ = results['logz'][-1]
    logZ_err = results['logzerr'][-1]

    print(f"\nlog(Evidence): {logZ:.2f} ± {logZ_err:.2f}")

    return results

def compute_bayes_factor(results_model1, results_model2):
    """
    Compute Bayes factor between two models
    BF = Z_model1 / Z_model2
    """
    logZ1 = results_model1['logz'][-1]
    logZ2 = results_model2['logz'][-1]

    log_BF = logZ1 - logZ2
    BF = np.exp(log_BF)

    print(f"\n{'='*70}")
    print(f"BAYES FACTOR ANALYSIS")
    print(f"{'='*70}")
    print(f"log(BF) = {log_BF:.2f}")
    print(f"BF = {BF:.2e}")

    # Jeffreys' scale interpretation
    if log_BF > 5:
        interpretation = "Decisive evidence for Model 1"
    elif 2.5 < log_BF <= 5:
        interpretation = "Strong evidence for Model 1"
    elif 1 < log_BF <= 2.5:
        interpretation = "Moderate evidence for Model 1"
    elif -1 < log_BF <= 1:
        interpretation = "Inconclusive"
    elif -2.5 < log_BF <= -1:
        interpretation = "Moderate evidence for Model 2"
    elif -5 < log_BF <= -2.5:
        interpretation = "Strong evidence for Model 2"
    else:
        interpretation = "Decisive evidence for Model 2"

    print(f"Interpretation (Jeffreys' scale): {interpretation}")

    return log_BF, BF

# ============================================================================
# VISUALIZATION
# ============================================================================
def plot_corner_posteriors(results, param_names, filename='corner_plot.png'):
    """
    Generate corner plot of posterior distributions
    """
    samples = results.samples
    weights = np.exp(results.logwt - results.logz[-1])

    fig = corner.corner(samples, labels=param_names, weights=weights,
                        quantiles=[0.16, 0.5, 0.84], show_titles=True,
                        title_fmt='.3f')

    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Corner plot saved: {filename}")
    plt.close()

def plot_data_vs_model(E_obs, t_obs, sigma_t, z, results, model_type,
                       filename='data_vs_model.png'):
    """
    Plot observed data vs best-fit model
    """
    # Get best-fit parameters (maximum likelihood)
    samples = results.samples
    weights = np.exp(results.logwt - results.logz[-1])

    # Compute weighted mean parameters
    theta_best = np.average(samples, weights=weights, axis=0)

    # Generate model predictions
    E_model = np.logspace(np.log10(E_obs.min()), np.log10(E_obs.max()), 100)

    if model_type == 'LIV_linear_SBPL':
        E_QG, zeta, E_b, alpha1, alpha2, mu, sign_raw = theta_best
        sign = 1 if sign_raw > 0.5 else -1
        params = {'E_QG': 10**E_QG, 'zeta': zeta, 'E_b': E_b,
                  'alpha1': alpha1, 'alpha2': alpha2, 'mu': mu, 'sign': sign}

        # Separate components
        t_LIV = sign * Delta_t_LIV_linear(E_model, z, 10**E_QG)
        t_int = Delta_t_intrinsic_SBPL(E_model, zeta, E_b, alpha1, alpha2, mu)
        t_total = t_LIV + t_int

    elif model_type == 'intrinsic_only_SBPL':
        zeta, E_b, alpha1, alpha2, mu = theta_best
        params = {'zeta': zeta, 'E_b': E_b, 'alpha1': alpha1,
                  'alpha2': alpha2, 'mu': mu}
        t_total = Delta_t_intrinsic_SBPL(E_model, zeta, E_b, alpha1, alpha2, mu)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.errorbar(E_obs, t_obs, yerr=sigma_t, fmt='o', label='Observed data',
                 alpha=0.6, capsize=3)
    plt.plot(E_model, t_total, 'r-', lw=2, label='Best-fit total model')

    if model_type == 'LIV_linear_SBPL':
        plt.plot(E_model, t_LIV, 'b--', lw=1.5, label='LIV component', alpha=0.7)
        plt.plot(E_model, t_int, 'g--', lw=1.5, label='Intrinsic component', alpha=0.7)

    plt.xlabel('Energy (GeV)', fontsize=12)
    plt.ylabel('Time delay (s)', fontsize=12)
    plt.xscale('log')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title(f'GRB 090510 (z={z:.3f}): Data vs Model', fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    print(f"Data vs Model plot saved: {filename}")
    plt.close()

# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("\n" + "="*70)
    print("ANÁLISIS BAYESIANO FERMI-LAT GRB: LORENTZ INVARIANCE VIOLATION")
    print("="*70)

    # Generate GRB 090510 mock data
    E_obs, t_obs, sigma_t, z = generate_GRB090510_mock_data()

    print("\n[1] ANÁLISIS MODELO LIV + SBPL INTRINSIC")
    results_LIV_SBPL = run_nested_sampling(E_obs, t_obs, sigma_t, z,
                                            model_type='LIV_linear_SBPL',
                                            nlive=300, dlogz=0.1)

    print("\n[2] ANÁLISIS MODELO INTRINSIC-ONLY (NULL HYPOTHESIS)")
    results_intrinsic = run_nested_sampling(E_obs, t_obs, sigma_t, z,
                                             model_type='intrinsic_only_SBPL',
                                             nlive=300, dlogz=0.1)

    # Compute Bayes Factor
    log_BF, BF = compute_bayes_factor(results_LIV_SBPL, results_intrinsic)

    # Generate plots
    param_names_LIV = [r'$\log_{10}(E_{QG}/{\rm GeV})$', r'$\zeta$ (s)',
                       r'$E_b$ (GeV)', r'$\alpha_1$', r'$\alpha_2$',
                       r'$\mu$', r'sign']
    param_names_int = [r'$\zeta$ (s)', r'$E_b$ (GeV)', r'$\alpha_1$',
                       r'$\alpha_2$', r'$\mu$']

    plot_corner_posteriors(results_LIV_SBPL, param_names_LIV,
                           'GRB090510_LIV_SBPL_corner.png')
    plot_corner_posteriors(results_intrinsic, param_names_int,
                           'GRB090510_intrinsic_corner.png')

    plot_data_vs_model(E_obs, t_obs, sigma_t, z, results_LIV_SBPL,
                       'LIV_linear_SBPL', 'GRB090510_LIV_model.png')
    plot_data_vs_model(E_obs, t_obs, sigma_t, z, results_intrinsic,
                       'intrinsic_only_SBPL', 'GRB090510_intrinsic_model.png')

    print("\n" + "="*70)
    print("ANÁLISIS COMPLETADO")
    print("="*70)


ANÁLISIS BAYESIANO FERMI-LAT GRB: LORENTZ INVARIANCE VIOLATION

[1] ANÁLISIS MODELO LIV + SBPL INTRINSIC

Running Nested Sampling: LIV_linear_SBPL
N_photons: 50
Redshift: 0.903
Energy range: 0.11 - 26.09 GeV


7813it [00:46, 168.02it/s, +300 | bound: 38 | nc: 1 | ncall: 192973 | eff(%):  4.211 | loglstar:   -inf <  8.088 <    inf | logz: -15.624 +/- 57.832 | dlogz:  0.000 >  0.100]



log(Evidence): -15.62 ± 0.30

[2] ANÁLISIS MODELO INTRINSIC-ONLY (NULL HYPOTHESIS)

Running Nested Sampling: intrinsic_only_SBPL
N_photons: 50
Redshift: 0.903
Energy range: 0.11 - 26.09 GeV


6348it [00:18, 342.79it/s, +300 | bound: 38 | nc: 1 | ncall: 142419 | eff(%):  4.678 | loglstar:   -inf < 25.486 <    inf | logz:  6.636 +/- 59.547 | dlogz:  0.000 >  0.100]



log(Evidence): 6.64 ± 0.26

BAYES FACTOR ANALYSIS
log(BF) = -22.26
BF = 2.15e-10
Interpretation (Jeffreys' scale): Decisive evidence for Model 2


Corner plot saved: GRB090510_LIV_SBPL_corner.png


Corner plot saved: GRB090510_intrinsic_corner.png
Data vs Model plot saved: GRB090510_LIV_model.png
Data vs Model plot saved: GRB090510_intrinsic_model.png

ANÁLISIS COMPLETADO


In [ ]:
!pip install corner

In [ ]:
!pip install dynesty

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.9/102.9 kB 4.3 MB/s eta 0:00:00
